# Explainable Temporal Graph Neural Networks for Adaptive Financial Fraud Detection
## End-to-End Training, Validation, and Benchmark Suite (Google Colab)

**Academic Honours Research Project**  
**Repository Reference:** `ETGNN-Fraud-Detection`  
**Dataset:** Elliptic Bitcoin Transaction Benchmark (203,769 nodes, 234,355 directed edges, 49 time steps)

---

### Research Objectives Covered in this Notebook
1. **Primary Baseline Benchmark:** Implement, train, and evaluate 7 core fraud detection models across chronological splits:
   - Traditional Tabular Classifiers: **Random Forest** (165f) & **XGBoost** (165f)
   - Standard Graph Neural Networks: **GCN**, **GraphSAGE**, and **GAT** (165f)
   - Proposed Architecture: **ETGNN** (165f + 16 sinusoidal time dimensions)
   - Exact Negative Control: **Matched Zero-Time GAT** (165f + 16 all-zero dimensions, parameter-matched at exactly 11,910 parameters)
2. **Strict Chronological Data Integrity:** 
   - Training: Time steps 1–32 (28,938 labelled transactions)
   - Validation: Time steps 33–40 (7,653 labelled transactions)
   - Held-Out Test: Time steps 41–49 (9,973 labelled transactions)
   - Preprocessing leakage prevention: Feature scaling fitted strictly on training data ($t \le 32$).
   - Unknown labels (label `-1`) retained strictly as structural message-passing context and excluded from supervised loss and evaluation.
3. **Explicit Execution Modes:**
   - `REAL_DATASET`: Trains all 7 models from scratch using the authentic Elliptic Bitcoin dataset.
   - `CHECKPOINT_EVALUATION`: Evaluates verified pre-trained checkpoints using the real dataset without retraining.
   - `SYNTHETIC_SMOKE_TEST`: Fast functional smoke-test on a synthetic graph (strictly marked **NOT RESEARCH RESULTS**).
4. **Historical Benchmark Comparison & Defensibility:**
   - Compares freshly generated metrics against historical repository results.
   - Acknowledges and documents tabular baseline dominance (Random Forest AUPRC 0.6323 vs. GNNs 0.1155–0.2427).


In [ ]:
# Section 1 & 2: Environment Setup, Dependency Installation & Google Drive Mounting
import os
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[Colab Setup] Running inside Google Colab environment.")
    print("[Colab Setup] Installing required packages...")
    !pip install -q torch_geometric xgboost joblib scikit-learn matplotlib seaborn tabulate
    
    # Optional Google Drive Mount
    MOUNT_GOOGLE_DRIVE = True  # Pre-configured: Automatically mounts Google Drive in Colab
    if MOUNT_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        print("[Colab Setup] Google Drive mounted at /content/drive")
else:
    print("[Local Setup] Running in local environment.")


In [ ]:
# Section 3: Hardware Verification, Device Selection & Seeding
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    device = torch.device("cuda")
    print(f"[Device] CUDA GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"[Device] VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    device = torch.device("cpu")
    print("[Device] CUDA not available. Running on CPU (training may take 10-15 minutes).")

print(f"[Torch] PyTorch version: {torch.__version__}")
try:
    import torch_geometric
    print(f"[PyG] PyTorch Geometric version: {torch_geometric.__version__}")
except ImportError:
    print("[PyG] PyTorch Geometric not installed. Run cell 2 to install.")


In [ ]:
# Section 4: Execution Configuration & Safe Output Directories
from datetime import datetime
from pathlib import Path

# =====================================================================
# USER CONFIGURATION:
# Select EXECUTION_MODE:
#   1. "REAL_DATASET"          - Train all 7 models from scratch on the real Elliptic dataset.
#   2. "CHECKPOINT_EVALUATION" - Evaluate existing checkpoints on real Elliptic data without retraining.
#   3. "SYNTHETIC_SMOKE_TEST"  - Fast functional test on a synthetic graph (NOT RESEARCH RESULTS).
# =====================================================================
EXECUTION_MODE = "REAL_DATASET"  # Options: "REAL_DATASET" | "CHECKPOINT_EVALUATION" | "SYNTHETIC_SMOKE_TEST"

# Dataset search paths (searches local directory, repository root, or Google Drive)
POSSIBLE_DATA_DIRS = [
    Path("data/processed"),
    Path("/content/data/processed"),
    Path("/content/drive/MyDrive/ETGNN/data/processed"),
    Path("/content/drive/MyDrive/ETGNN-Fraud-Detection/data/processed"),
    Path("/content/drive/MyDrive/data/processed"),
]

# Baseline historical checkpoints path (for CHECKPOINT_EVALUATION mode)
POSSIBLE_CHECKPOINT_DIRS = [
    Path("results/checkpoints"),
    Path("/content/results/checkpoints"),
    Path("/content/drive/MyDrive/ETGNN/results/checkpoints"),
    Path("/content/drive/MyDrive/ETGNN-Fraud-Detection/results/checkpoints"),
]

# Create timestamped run directory to prevent overwriting original checkpoints
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"results/colab_runs/run_{RUN_ID}")
OUTPUT_CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_TABLES_DIR = OUTPUT_DIR / "tables"

OUTPUT_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Config] Execution Mode: {EXECUTION_MODE}")
print(f"[Config] Timestamped Run Directory: {OUTPUT_DIR.resolve()}")
print(f"[Config] Safe Checkpoints Output:  {OUTPUT_CHECKPOINT_DIR.resolve()}")


In [ ]:
# Section 5: Real Dataset Ingestion & Integrity Verification (Never Silent Fallback)
import pandas as pd

def locate_data_dir(candidates):
    # 1. Check standard candidate directories first
    for p in candidates:
        if p.exists() and (p / "train_graph.pt").exists():
            return p
            
    # 2. If in Colab, automatically mount Drive if not already mounted
    if IN_COLAB and not Path("/content/drive/MyDrive").exists():
        try:
            print("[Colab Setup] Google Drive not detected at /content/drive. Mounting now...")
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print(f"[Colab Setup] Could not auto-mount Drive: {e}")
            
    # 3. Dynamic recursive search across Google Drive
    drive_mydrive = Path("/content/drive/MyDrive")
    if drive_mydrive.exists():
        print("[Dataset Discovery] Searching Google Drive for 'train_graph.pt'...")
        matches = list(drive_mydrive.glob("**/train_graph.pt"))
        if matches:
            found_dir = matches[0].parent
            print(f"[Dataset Discovery] Successfully found dataset at: {found_dir}")
            return found_dir
            
    # 4. Search directly under /content (in case files were uploaded directly)
    content_dir = Path("/content")
    if content_dir.exists():
        matches = [p for p in content_dir.glob("**/train_graph.pt") if not str(p).startswith("/content/drive")]
        if matches:
            found_dir = matches[0].parent
            print(f"[Dataset Discovery] Found dataset in local Colab space: {found_dir}")
            return found_dir

    return candidates[0]

DATA_DIR = locate_data_dir(POSSIBLE_DATA_DIRS)

def load_real_elliptic_dataset(data_dir: Path):
    required_files = {
        "train_graph.pt": data_dir / "train_graph.pt",
        "validation_graph.pt": data_dir / "validation_graph.pt",
        "test_graph.pt": data_dir / "test_graph.pt",
        "scaled_features.npy": data_dir / "scaled_features.npy",
        "nodes_with_masks.csv": data_dir / "nodes_with_masks.csv",
    }
    
    missing = [name for name, path in required_files.items() if not path.exists()]
    if missing:
        error_msg = (
            f"\n[ERROR] Missing required Elliptic dataset files in '{data_dir.resolve()}':\n"
            + "\n".join(f"  - {m}" for m in missing)
            + "\n\nTo resolve this in Google Colab:\n"
            + "  1. Upload the 'data/processed' folder to your Colab runtime or Google Drive.\n"
            + "  2. Or set MOUNT_GOOGLE_DRIVE = True in Cell 2 if data is stored in Drive.\n"
            + "  3. If you only want to test code execution without real data, switch to:\n"
            + "     EXECUTION_MODE = 'SYNTHETIC_SMOKE_TEST' in Cell 4.\n"
        )
        raise FileNotFoundError(error_msg)
        
    print(f"[Dataset] Loading authentic Elliptic dataset from: {data_dir.resolve()}")
    train_graph = torch.load(required_files["train_graph.pt"], map_location="cpu", weights_only=True)
    val_graph = torch.load(required_files["validation_graph.pt"], map_location="cpu", weights_only=True)
    test_graph = torch.load(required_files["test_graph.pt"], map_location="cpu", weights_only=True)
    nodes_df = pd.read_csv(required_files["nodes_with_masks.csv"])
    scaled_features = np.load(required_files["scaled_features.npy"], mmap_mode="r")
    
    # Rigorous dataset integrity checks
    assert len(nodes_df) == 203769, f"Expected 203,769 nodes, got {len(nodes_df)}"
    assert scaled_features.shape == (203769, 165), f"Expected shape (203769, 165), got {scaled_features.shape}"
    assert int(train_graph["train_mask"].sum()) == 28938, f"Expected 28,938 train nodes, got {int(train_graph['train_mask'].sum())}"
    assert int(val_graph["val_mask"].sum()) == 7653, f"Expected 7,653 val nodes, got {int(val_graph['val_mask'].sum())}"
    assert int(test_graph["test_mask"].sum()) == 9973, f"Expected 9,973 test nodes, got {int(test_graph['test_mask'].sum())}"
    
    return train_graph, val_graph, test_graph, nodes_df, scaled_features

def generate_synthetic_smoke_test_fixture():
    print("=" * 80)
    print("*** [WARNING: SYNTHETIC SMOKE-TEST DATASET — NOT RESEARCH RESULTS — FUNCTIONAL TEST ONLY] ***")
    print("=" * 80)
    N = 1200
    F_dim = 165
    timesteps = np.random.randint(1, 50, size=N)
    timesteps.sort()
    raw_labels = np.random.choice([1, 0, -1], size=N, p=[0.10, 0.40, 0.50])
    nodes_df = pd.DataFrame({"txId": np.arange(N), "time_step": timesteps, "label": raw_labels})
    labelled = nodes_df["label"] != -1
    nodes_df["train_mask"] = labelled & nodes_df["time_step"].between(1, 32)
    nodes_df["val_mask"] = labelled & nodes_df["time_step"].between(33, 40)
    nodes_df["test_mask"] = labelled & nodes_df["time_step"].between(41, 49)
    
    for mask_name in ["train_mask", "val_mask", "test_mask"]:
        idx = nodes_df[nodes_df[mask_name]].index
        if len(idx) > 0:
            nodes_df.loc[idx[0], "label"] = 1
            if len(idx) > 1:
                nodes_df.loc[idx[1], "label"] = 0
                
    scaled_features = np.random.randn(N, F_dim).astype(np.float32)
    illicit_idx = nodes_df[nodes_df["label"] == 1].index
    scaled_features[illicit_idx, :10] += 0.8
    
    src_list, dst_list = [], []
    for t in range(1, 50):
        t_nodes = np.where(timesteps == t)[0]
        if len(t_nodes) > 1:
            for _ in range(min(len(t_nodes) * 2, 20)):
                u, v = np.random.choice(t_nodes, size=2, replace=False)
                src_list.append(u); dst_list.append(v)
    all_edges = np.array([src_list, dst_list], dtype=np.int64) if src_list else np.empty((2, 0), dtype=np.int64)
    
    def build_subgraph(cutoff, mask_name):
        g_nodes = np.where(timesteps <= cutoff)[0]
        node_map = {old: new for new, old in enumerate(g_nodes)}
        if all_edges.shape[1] > 0:
            edge_mask = (timesteps[all_edges[0]] <= cutoff) & (timesteps[all_edges[1]] <= cutoff)
            sub_edges = all_edges[:, edge_mask]
            mapped_src = [node_map[u] for u in sub_edges[0]]
            mapped_dst = [node_map[v] for v in sub_edges[1]]
            edge_tensor = torch.tensor([mapped_src, mapped_dst], dtype=torch.long)
        else:
            edge_tensor = torch.empty((2, 0), dtype=torch.long)
            
        sub_df = nodes_df.iloc[g_nodes]
        return {
            "x": torch.from_numpy(scaled_features[g_nodes]),
            "edge_index": edge_tensor,
            "y": torch.tensor(sub_df["label"].to_numpy(), dtype=torch.long),
            "time_step": torch.tensor(sub_df["time_step"].to_numpy(), dtype=torch.long),
            "train_mask": torch.tensor(sub_df["train_mask"].to_numpy(), dtype=torch.bool),
            "val_mask": torch.tensor(sub_df["val_mask"].to_numpy(), dtype=torch.bool),
            "test_mask": torch.tensor(sub_df["test_mask"].to_numpy(), dtype=torch.bool),
            "original_node_index": torch.from_numpy(g_nodes)
        }
    return build_subgraph(32, "train_mask"), build_subgraph(40, "val_mask"), build_subgraph(49, "test_mask"), nodes_df, scaled_features

if EXECUTION_MODE in ["REAL_DATASET", "CHECKPOINT_EVALUATION"]:
    train_graph, val_graph, test_graph, nodes_df, scaled_features = load_real_elliptic_dataset(DATA_DIR)
    DATASET_TYPE = "REAL_ELLIPTIC_BITCOIN"
    print(f"[Dataset] Verified authentic Elliptic Dataset: 203,769 transactions, 49 time steps.")
else:
    train_graph, val_graph, test_graph, nodes_df, scaled_features = generate_synthetic_smoke_test_fixture()
    DATASET_TYPE = "SYNTHETIC_SMOKE_TEST"

print(f"[Dataset] Active Dataset Type: {DATASET_TYPE}")
print(f"[Dataset] Train Graph: {train_graph['x'].shape[0]:,} nodes, {train_graph['edge_index'].shape[1]:,} edges")
print(f"[Dataset] Val Graph:   {val_graph['x'].shape[0]:,} nodes, {val_graph['edge_index'].shape[1]:,} edges")
print(f"[Dataset] Test Graph:  {test_graph['x'].shape[0]:,} nodes, {test_graph['edge_index'].shape[1]:,} edges")
print(f"[Dataset] Train Mask Labelled: {int(train_graph['train_mask'].sum()):,}")
print(f"[Dataset] Val Mask Labelled:   {int(val_graph['val_mask'].sum()):,}")
print(f"[Dataset] Test Mask Labelled:  {int(test_graph['test_mask'].sum()):,}")


In [ ]:
# Section 6: Sinusoidal Temporal Encoding Definition
import math

def encode_time(time_step: torch.Tensor, dimension: int = 16) -> torch.Tensor:
    # Vaswani-style sinusoidal and cosinusoidal time projection.
    # Maps discrete integer time steps t in [1, 49] to a continuous 16-dimensional embedding.
    if dimension % 2 != 0:
        raise ValueError("dimension must be even")
    time = time_step.float().unsqueeze(1)
    frequency = torch.exp(
        torch.arange(0, dimension, 2, device=time.device).float()
        * (-math.log(10000.0) / dimension)
    )
    angles = time * frequency
    return torch.stack(
        (torch.sin(angles), torch.cos(angles)), dim=-1
    ).flatten(1)

# Verification of time encoding shape and properties
demo_times = torch.tensor([1, 10, 32, 49])
encoded_demo = encode_time(demo_times, dimension=16)
assert encoded_demo.shape == (4, 16), f"Expected shape (4, 16), got {encoded_demo.shape}"
print("[Time Encoding] Verified Vaswani sinusoidal projection: 4 steps -> shape (4, 16).")


In [ ]:
# Section 7: Neural Network Architecture Definitions
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

class GCNClassifier(nn.Module):
    # Two-layer Graph Convolutional Network (10,754 parameters)
    def __init__(self, input_dim=165, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class GraphSAGEClassifier(nn.Module):
    # Two-layer GraphSAGE with Mean Aggregation (21,442 parameters)
    def __init__(self, input_dim=165, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class GATClassifier(nn.Module):
    # Two-layer Graph Attention Network with 4 heads (10,886 parameters)
    def __init__(self, input_dim=165, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(input_dim, hidden_dim // 4, heads=4, dropout=dropout)
        self.conv2 = GATConv(hidden_dim, 2, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class ETGNNClassifier(nn.Module):
    # Proposed Time-Encoded Graph Attention Network (11,910 parameters)
    def __init__(self, input_dim=165, time_dim=16, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(input_dim + time_dim, hidden_dim // 4, heads=4, dropout=dropout)
        self.conv2 = GATConv(hidden_dim, 2, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout
        self.time_dim = time_dim

    def forward(self, x, edge_index, time_step):
        encoded = encode_time(time_step, self.time_dim)
        combined = torch.cat((x, encoded), dim=1)
        h = self.conv1(combined, edge_index).relu()
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

class MatchedGATClassifier(ETGNNClassifier):
    # Parameter-matched zero-time negative control (exactly 11,910 parameters)
    def forward(self, x, edge_index, time_step):
        zero_time = torch.zeros((x.shape[0], self.time_dim), dtype=x.dtype, device=x.device)
        combined = torch.cat((x, zero_time), dim=1)
        h = self.conv1(combined, edge_index).relu()
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

# Verify parameter counts
model_params = {
    "GCN": sum(p.numel() for p in GCNClassifier().parameters()),
    "GraphSAGE": sum(p.numel() for p in GraphSAGEClassifier().parameters()),
    "GAT": sum(p.numel() for p in GATClassifier().parameters()),
    "Matched GAT": sum(p.numel() for p in MatchedGATClassifier().parameters()),
    "Proposed ETGNN": sum(p.numel() for p in ETGNNClassifier().parameters()),
}
for name, count in model_params.items():
    print(f"[Architecture] {name:<16}: {count:>6,} trainable parameters")
assert model_params["Matched GAT"] == model_params["Proposed ETGNN"] == 11910, "Parameter mismatch!"


In [ ]:
# Section 8: Evaluation Metrics Definition
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

def evaluate_binary(y_true, probability, threshold=0.5):
    # Evaluates binary classification predictions using standard metrics:
    # AUPRC, F1, Precision, Recall, and ROC-AUC.
    predicted = (probability >= threshold).astype(int)
    return {
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probability)) if len(np.unique(y_true)) > 1 else 0.5,
        "auprc": float(average_precision_score(y_true, probability)),
    }


In [ ]:
# Section 9: Baseline 1 — Random Forest Training & Evaluation
import joblib
from sklearn.ensemble import RandomForestClassifier

def find_checkpoint(filename):
    for candidate_dir in POSSIBLE_CHECKPOINT_DIRS:
        p = candidate_dir / filename
        if p.exists():
            return p
    return None

rf_ckpt_existing = find_checkpoint("random_forest.joblib")
rf_save_path = OUTPUT_CHECKPOINT_DIR / "random_forest.joblib"

train_mask = nodes_df["train_mask"].to_numpy(dtype=bool)
val_mask = nodes_df["val_mask"].to_numpy(dtype=bool)

y_train = nodes_df.loc[train_mask, "label"].to_numpy()
y_val = nodes_df.loc[val_mask, "label"].to_numpy()

X_train = scaled_features[train_mask]
X_val = scaled_features[val_mask]

if EXECUTION_MODE == "CHECKPOINT_EVALUATION" and rf_ckpt_existing:
    print(f"[Random Forest] Loading verified checkpoint from {rf_ckpt_existing}...")
    rf_model = joblib.load(rf_ckpt_existing)
else:
    print(f"[Random Forest] Training on {len(X_train):,} labelled samples...")
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    )
    rf_model.fit(X_train, y_train)
    joblib.dump(rf_model, rf_save_path)
    print(f"[Random Forest] Model saved safely to {rf_save_path}")

rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
rf_val_scores = evaluate_binary(y_val, rf_val_prob)
print("[Random Forest] Validation Results (threshold=0.5):")
for k, v in rf_val_scores.items():
    print(f"  {k:<12}: {v:.4f}")


In [ ]:
# Section 10: Baseline 2 — XGBoost Training & Evaluation
from xgboost import XGBClassifier

xgb_ckpt_existing = find_checkpoint("xgboost.json")
xgb_save_path = OUTPUT_CHECKPOINT_DIR / "xgboost.json"

neg_count = (y_train == 0).sum()
pos_count = max((y_train == 1).sum(), 1)
scale_pos_weight = float(neg_count / pos_count)

if EXECUTION_MODE == "CHECKPOINT_EVALUATION" and xgb_ckpt_existing:
    print(f"[XGBoost] Loading verified checkpoint from {xgb_ckpt_existing}...")
    xgb_model = XGBClassifier(n_jobs=-1)
    xgb_model.load_model(xgb_ckpt_existing)
else:
    print(f"[XGBoost] Training on {len(X_train):,} samples (scale_pos_weight={scale_pos_weight:.2f})...")
    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        eval_metric="aucpr",
        early_stopping_rounds=20,
        n_jobs=-1,
        random_state=SEED,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_model.save_model(xgb_save_path)
    print(f"[XGBoost] Best iteration: {xgb_model.best_iteration}")
    print(f"[XGBoost] Model saved safely to {xgb_save_path}")

xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]
xgb_val_scores = evaluate_binary(y_val, xgb_val_prob)
print("[XGBoost] Validation Results (threshold=0.5):")
for k, v in xgb_val_scores.items():
    print(f"  {k:<12}: {v:.4f}")


In [ ]:
# Section 11: Generic GNN Training & Safe Checkpoint Selection Loop
def train_gnn(
    model_name: str,
    model: nn.Module,
    train_data: dict,
    val_data: dict,
    checkpoint_file: str,
    has_time: bool = False,
    max_epochs: int = 50,
    patience: int = 8,
    lr: float = 0.01,
    weight_decay: float = 5e-4
):
    # Standard early-stopping GNN training loop matching repository implementation:
    # - Optimizer: Adam (lr=0.01, weight_decay=5e-4)
    # - Loss: Weighted Cross-Entropy based on training prevalence ratio
    # - Early stopping: Validation AUPRC with patience=8 epochs
    # - Safe Saving: Saves to timestamped run folder to protect original checkpoints
    existing_ckpt = find_checkpoint(checkpoint_file)
    safe_save_path = OUTPUT_CHECKPOINT_DIR / checkpoint_file
    
    if EXECUTION_MODE == "CHECKPOINT_EVALUATION" and existing_ckpt:
        print(f"[{model_name}] Loading verified checkpoint from {existing_ckpt}...")
        model.load_state_dict(torch.load(existing_ckpt, map_location=device, weights_only=True))
        model.to(device)
        model.eval()
        with torch.no_grad():
            x_val = val_data["x"].to(device)
            edge_val = val_data["edge_index"].to(device)
            if has_time:
                t_val = val_data["time_step"].to(device)
                logits = model(x_val, edge_val, t_val)
            else:
                logits = model(x_val, edge_val)
            prob = torch.softmax(logits[val_data["val_mask"]], dim=1)[:, 1].cpu().numpy()
            y_v = val_data["y"][val_data["val_mask"]].cpu().numpy()
            scores = evaluate_binary(y_v, prob)
        print(f"[{model_name}] Checkpoint Val AUPRC: {scores['auprc']:.4f}")
        return model, scores

    print(f"[{model_name}] Training from scratch (max_epochs={max_epochs}, patience={patience})...")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    train_mask = train_data["train_mask"]
    train_y = train_data["y"][train_mask]
    n_neg = (train_y == 0).sum().item()
    n_pos = max((train_y == 1).sum().item(), 1)
    loss_weight = torch.tensor([1.0, float(n_neg / n_pos)], device=device)
    
    x_tr = train_data["x"].to(device)
    edge_tr = train_data["edge_index"].to(device)
    t_tr = train_data["time_step"].to(device) if has_time else None
    
    x_va = val_data["x"].to(device)
    edge_va = val_data["edge_index"].to(device)
    t_va = val_data["time_step"].to(device) if has_time else None
    val_mask = val_data["val_mask"]
    y_va_np = val_data["y"][val_mask].cpu().numpy()
    
    best_auprc = -1.0
    best_epoch = 0
    bad_epochs = 0
    history = []
    
    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        if has_time:
            logits = model(x_tr, edge_tr, t_tr)
        else:
            logits = model(x_tr, edge_tr)
            
        loss = F.cross_entropy(logits[train_mask], train_data["y"][train_mask].to(device), weight=loss_weight)
        loss.backward()
        optimizer.step()
        
        # Validation
        model.eval()
        with torch.no_grad():
            if has_time:
                va_logits = model(x_va, edge_va, t_va)
            else:
                va_logits = model(x_va, edge_va)
            va_prob = torch.softmax(va_logits[val_mask], dim=1)[:, 1].cpu().numpy()
            current_auprc = average_precision_score(y_va_np, va_prob)
            
        history.append({"epoch": epoch, "loss": loss.item(), "val_auprc": current_auprc})
        
        if current_auprc > best_auprc + 0.0001:
            best_auprc = current_auprc
            best_epoch = epoch
            bad_epochs = 0
            torch.save(model.state_dict(), safe_save_path)
        else:
            bad_epochs += 1
            
        if epoch == 1 or epoch % 5 == 0 or bad_epochs >= patience:
            print(f"  Epoch {epoch:02d}: loss={loss.item():.4f}, val_AUPRC={current_auprc:.4f} (best: {best_auprc:.4f} @ ep {best_epoch})")
            
        if bad_epochs >= patience:
            print(f"  Early stopping triggered at epoch {epoch}.")
            break
            
    # Load best checkpoint from safe run directory
    model.load_state_dict(torch.load(safe_save_path, map_location=device, weights_only=True))
    model.eval()
    with torch.no_grad():
        if has_time:
            final_logits = model(x_va, edge_va, t_va)
        else:
            final_logits = model(x_va, edge_va)
        best_prob = torch.softmax(final_logits[val_mask], dim=1)[:, 1].cpu().numpy()
        final_scores = evaluate_binary(y_va_np, best_prob)
        
    print(f"[{model_name}] Best Val AUPRC: {final_scores['auprc']:.4f} (saved to {safe_save_path.name})")
    return model, final_scores


In [ ]:
# Section 12: Baseline GNN Training (GCN, GraphSAGE, GAT)
print("=== 1. Training GCN ===")
gcn_model = GCNClassifier(input_dim=165, hidden_dim=64, dropout=0.3)
gcn_model, gcn_val_scores = train_gnn(
    "GCN", gcn_model, train_graph, val_graph, "gcn.pt", has_time=False
)

print("\n=== 2. Training GraphSAGE ===")
sage_model = GraphSAGEClassifier(input_dim=165, hidden_dim=64, dropout=0.3)
sage_model, sage_val_scores = train_gnn(
    "GraphSAGE", sage_model, train_graph, val_graph, "graphsage.pt", has_time=False
)

print("\n=== 3. Training GAT ===")
gat_model = GATClassifier(input_dim=165, hidden_dim=64, dropout=0.3)
gat_model, gat_val_scores = train_gnn(
    "GAT", gat_model, train_graph, val_graph, "gat.pt", has_time=False
)


In [ ]:
# Section 13: Negative Control & Proposed ETGNN Training
print("=== 4. Training Parameter-Matched Zero-Time GAT (Negative Control) ===")
mgat_model = MatchedGATClassifier(input_dim=165, time_dim=16, hidden_dim=64, dropout=0.3)
mgat_model, mgat_val_scores = train_gnn(
    "Matched GAT", mgat_model, train_graph, val_graph, "matched_gat.pt", has_time=True
)

print("\n=== 5. Training Proposed ETGNN (Time-Encoded GAT) ===")
etgnn_model = ETGNNClassifier(input_dim=165, time_dim=16, hidden_dim=64, dropout=0.3)
etgnn_model, etgnn_val_scores = train_gnn(
    "ETGNN", etgnn_model, train_graph, val_graph, "etgnn.pt", has_time=True
)


In [ ]:
# Section 14: Validation Set Benchmark Summary
val_summary = {
    "Random Forest": rf_val_scores,
    "XGBoost": xgb_val_scores,
    "GCN": gcn_val_scores,
    "GraphSAGE": sage_val_scores,
    "GAT": gat_val_scores,
    "Matched GAT": mgat_val_scores,
    "ETGNN": etgnn_val_scores,
}

df_val = pd.DataFrame.from_dict(val_summary, orient="index")
df_val.index.name = "Model"
if EXECUTION_MODE == "SYNTHETIC_SMOKE_TEST":
    print("*** [WARNING: SYNTHETIC DATASET — NOT RESEARCH RESULTS] ***")
print("=================== VALIDATION BENCHMARK RESULTS (Time Steps 33-40) ===================")
print(df_val[["auprc", "f1", "precision", "recall", "roc_auc"]].round(4).to_string())


In [ ]:
# Section 15: Held-Out Test Evaluation (Time Steps 41-49)
test_mask = nodes_df["test_mask"].to_numpy(dtype=bool)
y_test = nodes_df.loc[test_mask, "label"].to_numpy()
X_test = scaled_features[test_mask]

test_graph_mask = test_graph["test_mask"]
assert len(y_test) == int(test_graph_mask.sum()), "Test mask length mismatch"

test_results = {}
test_probabilities = {}

# 1. Random Forest
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]
test_probabilities["Random Forest"] = rf_test_prob
test_results["Random Forest"] = evaluate_binary(y_test, rf_test_prob)

# 2. XGBoost
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]
test_probabilities["XGBoost"] = xgb_test_prob
test_results["XGBoost"] = evaluate_binary(y_test, xgb_test_prob)

# Graph model evaluation helper
def eval_graph_test(name, model, has_time=False):
    model.eval()
    with torch.no_grad():
        x = test_graph["x"].to(device)
        edge = test_graph["edge_index"].to(device)
        if has_time:
            t = test_graph["time_step"].to(device)
            logits = model(x, edge, t)
        else:
            logits = model(x, edge)
        prob = torch.softmax(logits[test_graph_mask], dim=1)[:, 1].cpu().numpy()
        test_probabilities[name] = prob
        test_results[name] = evaluate_binary(y_test, prob)

# 3. GCN
eval_graph_test("GCN", gcn_model, has_time=False)
# 4. GraphSAGE
eval_graph_test("GraphSAGE", sage_model, has_time=False)
# 5. GAT
eval_graph_test("GAT", gat_model, has_time=False)
# 6. Matched GAT
eval_graph_test("Matched GAT", mgat_model, has_time=True)
# 7. Proposed ETGNN
eval_graph_test("ETGNN", etgnn_model, has_time=True)

df_test = pd.DataFrame.from_dict(test_results, orient="index")
df_test.index.name = "Model"

# Save results table to safe output directory
output_csv = OUTPUT_TABLES_DIR / "final_test_results_colab.csv"
df_test.to_csv(output_csv)

if EXECUTION_MODE == "SYNTHETIC_SMOKE_TEST":
    print("*** [WARNING: SYNTHETIC DATASET — NOT RESEARCH RESULTS] ***")
print("==================== FINAL HELD-OUT TEST RESULTS (Time Steps 41-49) ====================")
print(df_test[["auprc", "f1", "precision", "recall", "roc_auc"]].round(4).to_string())
print(f"\n[Test] Results saved to: {output_csv}")


In [ ]:
# Section 16: Historical Benchmark Comparison Table & Variance Analysis
historical_reference = {
    "Random Forest": 0.6323,
    "XGBoost": 0.5472,
    "GraphSAGE": 0.2427,
    "ETGNN": 0.2250,
    "Matched GAT": 0.2099,
    "GCN": 0.1306,
    "GAT": 0.1155,
}

df_comparison = df_test[["auprc"]].copy()
df_comparison.rename(columns={"auprc": "Current AUPRC"}, inplace=True)
df_comparison["Historical AUPRC"] = [historical_reference.get(m, np.nan) for m in df_comparison.index]
df_comparison["Difference"] = df_comparison["Current AUPRC"] - df_comparison["Historical AUPRC"]

print("==================== HISTORICAL BENCHMARK COMPARISON ====================")
if EXECUTION_MODE == "SYNTHETIC_SMOKE_TEST":
    print("*** [NOTE: Running on SYNTHETIC data — metric divergence from historical is expected] ***")
print(df_comparison.round(4).to_string())
print("""
[Scientific Integrity Note]:
Historical reference values are established baselines from fixed seeds (seed=42).
Minor differences in freshly trained GNN weights on GPU are normal due to:
1. PyTorch / CUDA atomic floating-point non-determinism during sparse message-passing.
2. Library versions (scikit-learn, XGBoost hist-tree algorithm).
3. Early-stopping epoch sensitivity on imbalanced validation sets.
""")


In [ ]:
# Section 17: Scientific Visualizations — Precision-Recall Curves & Confusion Matrices
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Precision-Recall Curves
ax_pr = axes[0]
colors = {
    "Random Forest": "#2ca02c",
    "XGBoost": "#1f77b4",
    "ETGNN": "#d62728",
    "Matched GAT": "#9467bd",
    "GraphSAGE": "#ff7f0e",
    "GCN": "#8c564b",
    "GAT": "#e377c2"
}

for name, prob in test_probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = test_results[name]["auprc"]
    ax_pr.plot(recall, precision, label=f"{name} (AUPRC = {ap:.4f})",
               color=colors.get(name, None), linewidth=2)

base_rate = float(y_test.mean())
ax_pr.axhline(base_rate, color="grey", linestyle="--", alpha=0.7, label=f"Base Rate ({base_rate:.4f})")

title_prefix = "[SYNTHETIC TEST] " if EXECUTION_MODE == "SYNTHETIC_SMOKE_TEST" else ""
ax_pr.set_title(f"{title_prefix}Held-Out Test Precision-Recall Curves (Steps 41-49)", fontsize=13, fontweight="bold")
ax_pr.set_xlabel("Recall", fontsize=11)
ax_pr.set_ylabel("Precision", fontsize=11)
ax_pr.set_xlim([0.0, 1.0])
ax_pr.set_ylim([0.0, 1.05])
ax_pr.grid(True, linestyle=":", alpha=0.6)
ax_pr.legend(loc="upper right", fontsize=9, framealpha=0.9)

# Plot 2: Confusion Matrix for Proposed ETGNN
ax_cm = axes[1]
et_pred = (test_probabilities["ETGNN"] >= 0.5).astype(int)
cm_et = confusion_matrix(y_test, et_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_et, display_labels=["Licit", "Illicit"])
disp.plot(ax=ax_cm, cmap="Blues", colorbar=False)
ax_cm.set_title(f"{title_prefix}Proposed ETGNN Confusion Matrix\n(F1={test_results['ETGNN']['f1']:.4f}, Recall={test_results['ETGNN']['recall']:.4f})",
                fontsize=13, fontweight="bold")

plt.tight_layout()
plot_path = OUTPUT_TABLES_DIR / "colab_benchmark_pr_curves.png"
plt.savefig(plot_path, dpi=300)
plt.show()
print(f"[Visualization] Figure saved safely to {plot_path}")


In [ ]:
# Section 18: Artifact Serialization & Export
np.savez_compressed(
    OUTPUT_TABLES_DIR / "colab_test_probabilities.npz",
    y_test=y_test,
    **{f"model_{name.replace(' ', '_')}": prob for name, prob in test_probabilities.items()}
)

# Export consolidated markdown report
report_lines = [
    "# ETGNN Google Colab Benchmark Verification Report",
    f"Run ID: `{RUN_ID}`",
    f"Execution Mode: `{EXECUTION_MODE}`",
    f"Dataset Type: `{DATASET_TYPE}`",
    f"Hardware Device: `{device}`",
    "",
    "## 1. Summary Results Table (Steps 41-49):",
    df_test.round(4).to_markdown(),
    "",
    "## 2. Historical Comparison Table:",
    df_comparison.round(4).to_markdown(),
    "",
    "## 3. Scientific Discussion & Academic Findings:",
    f"1. **Tabular Random Forest Dominance**: Random Forest achieved Test AUPRC of {test_results['Random Forest']['auprc']:.4f}, outperforming all graph neural network architectures.",
    f"2. **Proposed ETGNN Performance**: Achieved Test AUPRC of {test_results['ETGNN']['auprc']:.4f}.",
    f"3. **Parameter-Matched Control Contrast**: Matched Zero-Time GAT achieved {test_results['Matched GAT']['auprc']:.4f} vs ETGNN {test_results['ETGNN']['auprc']:.4f} (both at exactly 11,910 parameters)."
]
report_md = "\n".join(report_lines)
summary_path = OUTPUT_TABLES_DIR / "COLAB_BENCHMARK_SUMMARY.md"
summary_path.write_text(report_md, encoding="utf-8")
print(f"[Export] Consolidated report written to: {summary_path}")
